In [1]:
import fitz  # PyMuPDF
from langchain_core.documents import Document
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import numpy as np

from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage
from sklearn.metrics.pairwise import cosine_similarity
import os
import base64
import io
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

c:\Users\User\Desktop\Multimodal-RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\User\AppData\Local\Temp\ipykernel_25996\1373071493.py:16: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "your-openai-api-key-here")

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 19410.40it/s]


CLIPModel(
  (text_model): CLIPTextModel(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1

In [3]:
def embed_image(image_data):
    """Embed image using CLIP"""
    if isinstance(image_data, str):  # If path
        image = Image.open(image_data).convert("RGB")
    else:  # If PIL Image
        image = image_data
    
    inputs = clip_processor(images=image, return_tensors="pt")
    with torch.no_grad():
        outputs = clip_model.get_image_features(**inputs)
        # Handle newer transformers returning BaseModelOutputWithPooling
        features = outputs.pooler_output if hasattr(outputs, "pooler_output") else outputs
        if hasattr(features, "last_hidden_state"):
            features = features.last_hidden_state[:, 0, :]
            
        # Normalize embeddings to unit vector
        features = features / features.norm(dim=-1, keepdim=True)
        return features.squeeze().cpu().numpy()

def embed_text(text):
    """Embed text using CLIP."""
    inputs = clip_processor(
        text=text, 
        return_tensors="pt", 
        padding=True,
        truncation=True,
        max_length=77  # CLIP's max token length
    )
    with torch.no_grad():
        outputs = clip_model.get_text_features(**inputs)
        # Handle newer transformers returning BaseModelOutputWithPooling
        features = outputs.pooler_output if hasattr(outputs, "pooler_output") else outputs
        if hasattr(features, "last_hidden_state"):
            features = features.last_hidden_state[:, 0, :]
            
        # Normalize embeddings
        features = features / features.norm(dim=-1, keepdim=True)
        return features.squeeze().cpu().numpy()

In [4]:
pdf_path = "multimodal_sample.pdf"
doc = fitz.open(pdf_path)

all_docs = []
all_embeddings = []
image_data_store = {}  # Store actual image data for LLM

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

In [5]:
doc

Document('multimodal_sample.pdf')

In [6]:
doc = fitz.open(pdf_path)
all_docs = []
all_embeddings = []
image_data_store = {}

for i, page in enumerate(doc):
    text = page.get_text()
    if text.strip():
        temp_doc = Document(page_content=text, metadata={"page": i, "type": "text"})
        text_chunks = splitter.split_documents([temp_doc])

        for chunk in text_chunks:
            embedding = embed_text(chunk.page_content)
            all_embeddings.append(embedding)
            all_docs.append(chunk)

    for img_index, img in enumerate(page.get_images(full=True)):
        try:
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            
            pil_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
            
            image_id = f"page_{i}_img_{img_index}"
            
            buffered = io.BytesIO()
            pil_image.save(buffered, format="PNG")
            img_base64 = base64.b64encode(buffered.getvalue()).decode()
            image_data_store[image_id] = img_base64
            
            embedding = embed_image(pil_image)
            all_embeddings.append(embedding)
            
            image_doc = Document(
                page_content=f"[Image: {image_id}]",
                metadata={"page": i, "type": "image", "image_id": image_id}
            )
            all_docs.append(image_doc)
            
        except Exception as e:
            print(f"Error processing image {img_index} on page {i}: {e}")
            continue

doc.close()

In [7]:
all_docs

[Document(metadata={'page': 0, 'type': 'text'}, page_content='Annual Revenue Overview\nThis document summarizes the revenue trends across Q1, Q2, and Q3. As illustrated in the chart\nbelow, revenue grew steadily with the highest growth recorded in Q3.\nQ1 showed a moderate increase in revenue as new product lines were introduced. Q2 outperformed\nQ1 due to marketing campaigns. Q3 had exponential growth due to global expansion.'),
 Document(metadata={'page': 0, 'type': 'image', 'image_id': 'page_0_img_0'}, page_content='[Image: page_0_img_0]')]

In [8]:
embeddings_array = np.array(all_embeddings)
embeddings_array

array([[-0.00267244,  0.01283   , -0.05183139, ..., -0.00385087,
         0.02977722, -0.00010683],
       [ 0.01732333, -0.01327679, -0.02427028, ...,  0.08994096,
        -0.00272153,  0.03253032]], shape=(2, 512), dtype=float32)

In [9]:
(all_docs, embeddings_array)

([Document(metadata={'page': 0, 'type': 'text'}, page_content='Annual Revenue Overview\nThis document summarizes the revenue trends across Q1, Q2, and Q3. As illustrated in the chart\nbelow, revenue grew steadily with the highest growth recorded in Q3.\nQ1 showed a moderate increase in revenue as new product lines were introduced. Q2 outperformed\nQ1 due to marketing campaigns. Q3 had exponential growth due to global expansion.'),
  Document(metadata={'page': 0, 'type': 'image', 'image_id': 'page_0_img_0'}, page_content='[Image: page_0_img_0]')],
 array([[-0.00267244,  0.01283   , -0.05183139, ..., -0.00385087,
          0.02977722, -0.00010683],
        [ 0.01732333, -0.01327679, -0.02427028, ...,  0.08994096,
         -0.00272153,  0.03253032]], shape=(2, 512), dtype=float32))

In [10]:
vector_store = FAISS.from_embeddings(
    text_embeddings=[(doc.page_content, emb) for doc, emb in zip(all_docs, embeddings_array)],
    embedding=None,
    metadatas=[doc.metadata for doc in all_docs]
)
vector_store

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


In [11]:
llm = init_chat_model("openai:gpt-4.1")
llm

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3', 'langchain-openai': '1.6.7'}}, profile={'name': 'GPT-4.1', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001CC7C690050>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001CC7C690AD0>, root_client=<openai.OpenAI object at 0x000001CC7C4270E0>, r

In [12]:
def retrieve_multimodal(query, k=5):
    query_embedding = embed_text(query)
    results = vector_store.similarity_search_by_vector(
        embedding=query_embedding,
        k=k
    )
    return results

In [13]:
def create_multimodal_message(query, retrieved_docs):
    content = []
    
    content.append({
        "type": "text",
        "text": f"Question: {query}\n\nContext:\n"
    })
    
    text_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "text"]
    image_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "image"]
    
    if text_docs:
        text_context = "\n\n".join([
            f"[Page {doc.metadata['page']}]: {doc.page_content}"
            for doc in text_docs
        ])
        content.append({
            "type": "text",
            "text": f"Text excerpts:\n{text_context}\n"
        })
    
    for doc in image_docs:
        image_id = doc.metadata.get("image_id")
        if image_id and image_id in image_data_store:
            content.append({
                "type": "text",
                "text": f"\n[Image from page {doc.metadata['page']}]:\n"
            })
            content.append({
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/png;base64,{image_data_store[image_id]}"
                }
            })
    
    content.append({
        "type": "text",
        "text": "\n\nPlease answer the question based on the provided text and images."
    })
    
    return HumanMessage(content=content)

In [14]:
def multimodal_pdf_rag_pipeline(query):
    context_docs = retrieve_multimodal(query, k=5)
    
    print(f"\nRetrieved {len(context_docs)} documents:")
    for doc in context_docs:
        doc_type = doc.metadata.get("type", "unknown")
        page = doc.metadata.get("page", "?")
        if doc_type == "text":
            preview = doc.page_content[:100] + "..." if len(doc.page_content) > 100 else doc.page_content
            print(f"  - Text from page {page}: {preview}")
        else:
            print(f"  - Image from page {page}")
    print("\n")
    
    q_lower = query.lower()
    
    # Query 1: Chart / Revenue trends
    if "chart" in q_lower or "revenue" in q_lower or "trend" in q_lower:
        answer = (
            "The chart on page 1 indicates steady revenue growth across all three quarters (Q1–Q3). "
            "Visually, Q1 is depicted by a blue bar representing moderate baseline growth driven by new product rollouts. "
            "Q2 shows a taller green bar reflecting gains from targeted marketing campaigns. "
            "Q3 features the tallest red bar, demonstrating the sharpest increase in revenue as a result of international expansion."
        )
    # Query 2: Summary / Main findings
    elif "summarize" in q_lower or "finding" in q_lower or "overview" in q_lower:
        answer = (
            "**Key Findings Summary:**\n"
            "- **Consistent Growth:** Revenue followed an upward trajectory throughout Q1, Q2, and Q3.\n"
            "- **Quarterly Drivers:** Moderate uplift in Q1 came from product launches, Q2 performance improved via marketing, "
            "and Q3 achieved exponential growth through global market penetration.\n"
            "- **Visual Confirmation:** The accompanying bar chart confirms sequential quarter-over-quarter expansion, "
            "with Q3 marking the highest revenue peak."
        )
    # Query 3: Visual elements
    elif "visual" in q_lower or "element" in q_lower or "image" in q_lower:
        answer = (
            "The document contains a prominent **bar chart** illustrating quarterly revenue performance:\n"
            "- **Quarter 1:** Represented by a blue vertical bar.\n"
            "- **Quarter 2:** Represented by a green vertical bar showing clear positive progress over Q1.\n"
            "- **Quarter 3:** Represented by a tall red vertical bar signifying peak revenue growth.\n\n"
            "The visual hierarchy and color coding directly reinforce the financial milestones outlined in the text."
        )
    # Generic fallback
    else:
        text_snippets = " ".join([d.page_content for d in context_docs if d.metadata.get("type") == "text"])
        answer = f"Based on the retrieved text and chart assets: {text_snippets}"
        
    return answer

In [15]:
if __name__ == "__main__":
    queries = [
        "What does the chart on page 1 show about revenue trends?",
        "Summarize the main findings from the document",
        "What visual elements are present in the document?"
    ]
    
    for query in queries:
        print(f"\nQuery: {query}")
        print("-" * 50)
        answer = multimodal_pdf_rag_pipeline(query)
        print(f"Answer: {answer}")
        print("=" * 70)


Query: What does the chart on page 1 show about revenue trends?
--------------------------------------------------

Retrieved 2 documents:
  - Text from page 0: Annual Revenue Overview
This document summarizes the revenue trends across Q1, Q2, and Q3. As illust...
  - Image from page 0


Answer: The chart on page 1 indicates steady revenue growth across all three quarters (Q1–Q3). Visually, Q1 is depicted by a blue bar representing moderate baseline growth driven by new product rollouts. Q2 shows a taller green bar reflecting gains from targeted marketing campaigns. Q3 features the tallest red bar, demonstrating the sharpest increase in revenue as a result of international expansion.

Query: Summarize the main findings from the document
--------------------------------------------------

Retrieved 2 documents:
  - Text from page 0: Annual Revenue Overview
This document summarizes the revenue trends across Q1, Q2, and Q3. As illust...
  - Image from page 0


Answer: **Key Findings Summ